# MV CAF Backbone Test v1.1.1

`MVCAF_backbone_test_v1.1`의 규격을 유지하면서 DINOv2 backbone 추가 실험과 기존 v1.1 결과 병합 ensemble을 수행하는 notebook입니다. dev 기준 early stopping, TTA, test inference, v1.1 결합 평가까지 포함합니다.


In [ ]:
from __future__ import annotations

import json
import os
import shutil
import sys
from dataclasses import dataclass
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import timm
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm

ROOT = Path.cwd().resolve().parent
SRC_DIR = ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from augmentations import build_default_transforms
from models import EMAConfig, ModelEMA
from reproducibility import make_generator, seed_everything, seed_worker

DATA_DIR = ROOT / "data"
EXPERIMENT_NAME = "mv_caf_backbone_test"
PREV_VERSION = "v1.1"
CUR_VERSION = "v1.1.1"

WEIGHT_DIR = ROOT / "outputs" / "weights" / EXPERIMENT_NAME / CUR_VERSION
SUBMISSION_DIR = ROOT / "outputs" / "submissions" / EXPERIMENT_NAME / CUR_VERSION
EDA_DIR = ROOT / "outputs" / "eda_preprocessing" / EXPERIMENT_NAME / CUR_VERSION
PREV_SUBMISSION_DIR = ROOT / "outputs" / "submissions" / EXPERIMENT_NAME / PREV_VERSION
PREV_EDA_DIR = ROOT / "outputs" / "eda_preprocessing" / EXPERIMENT_NAME / PREV_VERSION

WEIGHT_DIR.mkdir(parents=True, exist_ok=True)
SUBMISSION_DIR.mkdir(parents=True, exist_ok=True)
EDA_DIR.mkdir(parents=True, exist_ok=True)

CFG = {
    "IMG_SIZE": 224,
    "EPOCHS": 30,
    "EARLY_STOPPING_PATIENCE": 5,
    "LEARNING_RATE": 3e-4,
    "BATCH_SIZE": 8,
    "SEED": 42,
    "NUM_WORKERS": 8,
    "WEIGHT_DECAY": 1e-4,
    "MIXUP_ALPHA": 0.1,
    "MIXUP_PROB": 0.0,
    "MIN_LR": 1e-6,
    "EMA_DECAY": 0.999,
    "EMA_USE_FOR_EVAL": True,
    "PRETRAINED": True,
    "BASE_BATCH_SIZE": 16,
    "MIN_BATCH_SIZE": 1,
    "USE_AMP": True,
    "TTA_CANDIDATES": [
        ["none"],
        ["none", "hflip"],
        ["none", "hflip", "crop95"],
    ],
    "VIDEO_AUG_ENABLE": True,
    "VIDEO_AUG_CACHE": True,
    "UNSTABLE_START_MIN_SEC": 0.5,
    "UNSTABLE_START_MAX_SEC": 1.0,
    "UNSTABLE_FRAMES_MIN": 2,
    "UNSTABLE_FRAMES_MAX": 3,
    "STABLE_END_MIN_SEC": 9.0,
    "STABLE_END_MAX_SEC": 10.0,
    "STABLE_FRAMES_PER_VIDEO": 2,
}

BACKBONE_CANDIDATES = [
    "vit_base_patch14_dinov2.lvd142m",
]

selected_backbones = []
for name in BACKBONE_CANDIDATES:
    try:
        timm.create_model(name, pretrained=False, num_classes=0, global_pool="")
        selected_backbones.append(name)
    except Exception as exc:
        print("skip backbone:", name, exc)

assert selected_backbones, "No candidate backbones are available in this timm installation."


def resolve_backbone_img_size(backbone_name: str) -> int:
    backbone = timm.create_model(backbone_name, pretrained=False, num_classes=0, global_pool="")
    cfg = getattr(backbone, "pretrained_cfg", None) or getattr(backbone, "default_cfg", {}) or {}
    input_size = cfg.get("input_size") or (3, CFG["IMG_SIZE"], CFG["IMG_SIZE"])
    return int(input_size[-1])


def resolve_backbone_batch_size(img_size: int) -> int:
    scaled = int(CFG["BASE_BATCH_SIZE"] * (CFG["IMG_SIZE"] / img_size) ** 2)
    return max(CFG["MIN_BATCH_SIZE"], min(CFG["BASE_BATCH_SIZE"], scaled))


BACKBONE_IMG_SIZES = {name: resolve_backbone_img_size(name) for name in selected_backbones}
BACKBONE_BATCH_SIZES = {name: resolve_backbone_batch_size(img_size) for name, img_size in BACKBONE_IMG_SIZES.items()}

seed_everything(CFG["SEED"])
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
amp_enabled = bool(CFG["USE_AMP"] and device.type == "cuda")
amp_dtype = torch.bfloat16 if amp_enabled and torch.cuda.is_bf16_supported() else torch.float16
print("device:", device)
print("selected_backbones:", selected_backbones)
print("backbone_img_sizes:", BACKBONE_IMG_SIZES)
print("backbone_batch_sizes:", BACKBONE_BATCH_SIZES)


device: cuda
selected_backbones: ['vit_base_patch14_dinov2.lvd142m']
backbone_img_sizes: {'vit_base_patch14_dinov2.lvd142m': 518}


In [2]:
class MultiViewDataset(Dataset):
    def __init__(self, df, root_dir, transform=None, is_test=False):
        self.df = df.reset_index(drop=True)
        self.root_dir = root_dir
        self.transform = transform
        self.is_test = is_test
        self.label_map = {"stable": 0, "unstable": 1}

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        sample_id = str(row["id"])
        base_dir = self.root_dir
        if ("sample_dir" in self.df.columns) and pd.notna(row.get("sample_dir", np.nan)):
            base_dir = str(row["sample_dir"])
        folder_path = os.path.join(base_dir, sample_id)

        views = []
        for name in ["front", "top"]:
            img_path = os.path.join(folder_path, f"{name}.png")
            image = Image.open(img_path).convert("RGB")
            image = self.transform(image) if self.transform else image
            views.append(image)

        if self.is_test:
            return views, sample_id
        label = self.label_map[str(row["label"])]
        return views, torch.tensor(label, dtype=torch.float32)


def _video_aug_cache_signature(cfg):
    keys = [
        "SEED",
        "UNSTABLE_START_MIN_SEC",
        "UNSTABLE_START_MAX_SEC",
        "UNSTABLE_FRAMES_MIN",
        "UNSTABLE_FRAMES_MAX",
        "STABLE_END_MIN_SEC",
        "STABLE_END_MAX_SEC",
        "STABLE_FRAMES_PER_VIDEO",
    ]
    return json.dumps({k: cfg[k] for k in keys}, sort_keys=True)


def build_video_augmented_df(train_df: pd.DataFrame, data_dir: Path, cfg: dict) -> pd.DataFrame:
    video_dir = data_dir / "train_video"
    aug_root = data_dir / "train_video_aug"
    aug_root.mkdir(parents=True, exist_ok=True)

    cache_csv = aug_root / "aug_df.csv"
    cache_meta = aug_root / "cache_meta.json"
    cache_sig = _video_aug_cache_signature(cfg)

    if cfg.get("VIDEO_AUG_CACHE", True) and cache_csv.exists() and cache_meta.exists():
        try:
            meta = json.loads(cache_meta.read_text())
            if meta.get("signature") == cache_sig:
                cached_df = pd.read_csv(cache_csv)
                if not cached_df.empty:
                    cached_df["sample_dir"] = str(aug_root)
                    return cached_df
        except Exception as exc:
            print("video aug cache read failed:", exc)

    for p in aug_root.glob("AUGV_*"):
        if p.is_dir():
            shutil.rmtree(p, ignore_errors=True)

    rng = np.random.default_rng(cfg["SEED"])
    stable_rows = []
    unstable_rows = []
    saved_idx = 0

    def save_aug(img, label):
        nonlocal saved_idx
        aug_id = f"AUGV_{saved_idx:07d}"
        out_dir = aug_root / aug_id
        out_dir.mkdir(parents=True, exist_ok=True)
        Image.fromarray(img).save(out_dir / "front.png")
        Image.fromarray(img).save(out_dir / "top.png")
        row = {"id": aug_id, "label": label, "sample_dir": str(aug_root)}
        saved_idx += 1
        return row

    def read_frames(video_path: Path):
        cap = cv2.VideoCapture(str(video_path))
        fps = cap.get(cv2.CAP_PROP_FPS)
        if fps <= 0:
            fps = 30.0
        frames = []
        while True:
            ok, frame = cap.read()
            if not ok:
                break
            frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        cap.release()
        return frames, fps

    for _, row in tqdm(train_df.iterrows(), total=len(train_df), desc="video aug", dynamic_ncols=True, ascii=True):
        sample_id = str(row["id"])
        label = str(row["label"])
        video_path = video_dir / f"{sample_id}.mp4"
        if not video_path.exists():
            continue

        frames, fps = read_frames(video_path)
        if not frames:
            continue

        n_frames = len(frames)
        if label == "unstable":
            start_sec = rng.uniform(cfg["UNSTABLE_START_MIN_SEC"], cfg["UNSTABLE_START_MAX_SEC"])
            n_pick = int(rng.integers(cfg["UNSTABLE_FRAMES_MIN"], cfg["UNSTABLE_FRAMES_MAX"] + 1))
            start_idx = min(n_frames - 1, max(0, int(start_sec * fps)))
            end_idx = min(n_frames, start_idx + max(1, n_pick))
            for frame in frames[start_idx:end_idx]:
                unstable_rows.append(save_aug(frame, label))
        else:
            start_sec = rng.uniform(cfg["STABLE_END_MIN_SEC"], cfg["STABLE_END_MAX_SEC"])
            n_pick = int(cfg["STABLE_FRAMES_PER_VIDEO"])
            start_idx = min(n_frames - 1, max(0, int(start_sec * fps)))
            indices = np.linspace(start_idx, n_frames - 1, num=n_pick, dtype=int)
            for idx in indices:
                stable_rows.append(save_aug(frames[int(idx)], label))

    aug_df = pd.DataFrame(stable_rows + unstable_rows)
    if aug_df.empty:
        return aug_df
    aug_df.to_csv(cache_csv, index=False)
    cache_meta.write_text(json.dumps({"signature": cache_sig}, ensure_ascii=False, indent=2))
    return aug_df


In [3]:
@dataclass
class FlexibleCAFConfig:
    backbone_name: str
    attn_heads: int = 8
    dropout: float = 0.1


class MultiViewFlexibleCAF(nn.Module):
    def __init__(self, config: FlexibleCAFConfig):
        super().__init__()
        self.backbone = timm.create_model(
            config.backbone_name,
            pretrained=CFG["PRETRAINED"],
            num_classes=0,
            global_pool="",
        )
        self.embed_dim = int(getattr(self.backbone, "num_features"))
        self.norm = nn.LayerNorm(self.embed_dim)
        self.attn = nn.MultiheadAttention(
            embed_dim=self.embed_dim,
            num_heads=config.attn_heads,
            dropout=config.dropout,
            batch_first=True,
        )
        self.head = nn.Sequential(
            nn.LayerNorm(self.embed_dim),
            nn.Dropout(config.dropout),
            nn.Linear(self.embed_dim, 1),
        )

    def _encode_single(self, x: torch.Tensor) -> torch.Tensor:
        feat = self.backbone.forward_features(x)
        if isinstance(feat, dict):
            feat = next(v for v in reversed(list(feat.values())) if isinstance(v, torch.Tensor))
        if isinstance(feat, (list, tuple)):
            feat = feat[-1]
        if feat.ndim == 4:
            feat = feat.mean(dim=(2, 3))
        elif feat.ndim == 3:
            feat = feat.mean(dim=1)
        return feat

    def forward(self, views):
        tokens = torch.stack([self._encode_single(v) for v in views], dim=1)
        tokens = self.norm(tokens)
        fused, _ = self.attn(tokens, tokens, tokens, need_weights=False)
        fused = fused.mean(dim=1)
        return self.head(fused)


In [4]:
train_df = pd.read_csv(DATA_DIR / "train.csv", encoding="utf-8-sig")
val_df = pd.read_csv(DATA_DIR / "dev.csv", encoding="utf-8-sig")
test_df = pd.read_csv(DATA_DIR / "sample_submission.csv", encoding="utf-8-sig")

train_df_for_train = train_df.copy()
train_df_for_train["sample_dir"] = str(DATA_DIR / "train")

if CFG["VIDEO_AUG_ENABLE"]:
    aug_df = build_video_augmented_df(train_df, DATA_DIR, CFG)
    if not aug_df.empty:
        train_df_for_train = pd.concat([train_df_for_train, aug_df], ignore_index=True)

val_df_for_eval = val_df.copy()
val_df_for_eval["sample_dir"] = str(DATA_DIR / "dev")
test_df_for_infer = test_df.copy()
test_df_for_infer["sample_dir"] = str(DATA_DIR / "test")

def build_dataloaders(img_size: int, batch_size: int):
    train_transform, test_transform = build_default_transforms(img_size)

    train_dataset = MultiViewDataset(train_df_for_train, str(DATA_DIR / "train"), train_transform, is_test=False)
    val_dataset = MultiViewDataset(val_df_for_eval, str(DATA_DIR / "dev"), test_transform, is_test=False)
    test_dataset = MultiViewDataset(test_df_for_infer, str(DATA_DIR / "test"), test_transform, is_test=True)

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=CFG["NUM_WORKERS"],
        pin_memory=(device.type == "cuda"),
        worker_init_fn=seed_worker,
        generator=make_generator(CFG["SEED"]),
    )
    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=CFG["NUM_WORKERS"],
        pin_memory=(device.type == "cuda"),
        worker_init_fn=seed_worker,
        generator=make_generator(CFG["SEED"] + 1),
    )
    test_loader = DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=CFG["NUM_WORKERS"],
        pin_memory=(device.type == "cuda"),
        worker_init_fn=seed_worker,
        generator=make_generator(CFG["SEED"] + 2),
    )
    return train_loader, val_loader, test_loader


video aug: 100%|##########| 1000/1000 [00:00<00:00, 36939.25it/s]


In [5]:
def mixup_multiview_batch(views, labels, alpha=0.2):
    if alpha <= 0:
        return views, labels, labels, 1.0
    lam = np.random.beta(alpha, alpha)
    index = torch.randperm(labels.size(0), device=labels.device)
    mixed_views = [lam * v + (1.0 - lam) * v[index, :] for v in views]
    return mixed_views, labels, labels[index], lam


def train_one_epoch(
    model,
    loader,
    criterion,
    optimizer,
    device,
    mixup_alpha=0.2,
    mixup_prob=0.5,
    ema=None,
    scaler=None,
    accum_steps=1,
):
    model.train()
    train_loss = 0.0
    optimizer.zero_grad(set_to_none=True)
    for step_idx, (views, labels) in enumerate(tqdm(loader, desc="Training", dynamic_ncols=True, ascii=True), start=1):
        views = [v.to(device) for v in views]
        labels = labels.to(device).float()

        with torch.autocast(device_type=device.type, dtype=amp_dtype, enabled=amp_enabled):
            if mixup_alpha > 0 and np.random.rand() < mixup_prob:
                mixed_views, labels_a, labels_b, lam = mixup_multiview_batch(views, labels, alpha=mixup_alpha)
                outputs = model(mixed_views).view(-1)
                loss = lam * criterion(outputs, labels_a) + (1.0 - lam) * criterion(outputs, labels_b)
            else:
                outputs = model(views).view(-1)
                loss = criterion(outputs, labels)

        scaled_loss = loss / accum_steps
        if scaler is not None:
            scaler.scale(scaled_loss).backward()
        else:
            scaled_loss.backward()

        should_step = (step_idx % accum_steps == 0) or (step_idx == len(loader))
        if should_step:
            if scaler is not None:
                scaler.step(optimizer)
                scaler.update()
            else:
                optimizer.step()
            optimizer.zero_grad(set_to_none=True)
            if ema is not None:
                ema.update(model)

        train_loss += float(loss.item()) * labels.size(0)
    return train_loss / max(1, len(loader.dataset))


@torch.no_grad()
def predict_probs(model, loader, device, tta_names=None):
    model.eval()
    probs = []
    labels = []
    ids = []
    tta_names = tta_names or ["none"]
    for batch in tqdm(loader, desc="Infer", dynamic_ncols=True, ascii=True):
        views, target = batch
        views = [v.to(device) for v in views]
        batch_probs = []
        for tta_name in tta_names:
            cur_views = views
            if tta_name == "hflip":
                cur_views = [torch.flip(v, dims=[3]) for v in views]
            elif tta_name == "crop95":
                cur_views = []
                for v in views:
                    _, _, h, w = v.shape
                    dh = max(1, int(h * 0.025))
                    dw = max(1, int(w * 0.025))
                    cropped = v[:, :, dh:h - dh, dw:w - dw]
                    cropped = F.interpolate(cropped, size=(h, w), mode="bilinear", align_corners=False)
                    cur_views.append(cropped)
            with torch.autocast(device_type=device.type, dtype=amp_dtype, enabled=amp_enabled):
                logits = model(cur_views).view(-1)
            batch_probs.append(torch.sigmoid(logits).cpu().numpy())
        probs.append(np.mean(batch_probs, axis=0))
        if isinstance(target, tuple):
            ids.extend(list(target))
        else:
            labels.extend(target.numpy().tolist())
    probs = np.concatenate(probs)
    return probs, np.array(labels), ids


@torch.no_grad()
def validate(model, loader, device):
    probs, labels, _ = predict_probs(model, loader, device, tta_names=["none"])
    probs = np.clip(probs, 1e-6, 1 - 1e-6)
    logloss = float(-np.mean(labels * np.log(probs) + (1 - labels) * np.log(1 - probs)))
    acc = float(np.mean((probs > 0.5) == labels))
    return logloss, acc, probs, labels


@torch.no_grad()
def evaluate_tta_grid(model, loader, device, candidates):
    rows = []
    for tta_names in candidates:
        probs, labels, _ = predict_probs(model, loader, device, tta_names=tta_names)
        p = np.clip(probs, 1e-6, 1 - 1e-6)
        rows.append({
            "tta_names": tta_names,
            "val_logloss": float(-np.mean(labels * np.log(p) + (1 - labels) * np.log(1 - p))),
            "val_acc": float(np.mean((probs > 0.5) == labels)),
        })
    return pd.DataFrame(rows).sort_values("val_logloss").reset_index(drop=True)


In [6]:
def train_single_backbone(backbone_name: str):
    img_size = BACKBONE_IMG_SIZES[backbone_name]
    batch_size = BACKBONE_BATCH_SIZES[backbone_name]
    accum_steps = max(1, int(np.ceil(CFG["BASE_BATCH_SIZE"] / batch_size)))
    train_loader, val_loader, test_loader = build_dataloaders(img_size, batch_size)
    scaler = torch.amp.GradScaler("cuda", enabled=amp_enabled and amp_dtype == torch.float16)
    print(f"[{backbone_name}] using img_size={img_size}, batch_size={batch_size}, accum_steps={accum_steps}, amp={amp_enabled}")

    config = FlexibleCAFConfig(backbone_name=backbone_name)
    model = MultiViewFlexibleCAF(config).to(device)
    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.Adam(model.parameters(), lr=CFG["LEARNING_RATE"], weight_decay=CFG["WEIGHT_DECAY"])
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CFG["EPOCHS"], eta_min=CFG["MIN_LR"])
    ema = ModelEMA(model, EMAConfig(decay=CFG["EMA_DECAY"]))

    safe_name = backbone_name.replace(".", "_").replace("/", "_")
    checkpoint_path = WEIGHT_DIR / f"mv_caf_backbone_test_{safe_name}_{CUR_VERSION}.pt"

    best_logloss = float("inf")
    best_acc = 0.0
    patience_left = CFG["EARLY_STOPPING_PATIENCE"]
    history = []
    if device.type == "cuda":
        torch.cuda.empty_cache()

    for epoch in range(1, CFG["EPOCHS"] + 1):
        train_loss = train_one_epoch(
            model,
            train_loader,
            criterion,
            optimizer,
            device,
            mixup_alpha=CFG["MIXUP_ALPHA"],
            mixup_prob=CFG["MIXUP_PROB"],
            ema=ema,
            scaler=scaler,
            accum_steps=accum_steps,
        )
        eval_model = ema.ema_model if CFG["EMA_USE_FOR_EVAL"] else model
        val_logloss, val_acc, _, _ = validate(eval_model, val_loader, device)

        improved = val_logloss < best_logloss
        if improved:
            best_logloss = val_logloss
            best_acc = val_acc
            patience_left = CFG["EARLY_STOPPING_PATIENCE"]
            torch.save({
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "ema_state_dict": ema.ema_model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "cfg": CFG,
                "backbone_name": backbone_name,
                "val_logloss": val_logloss,
                "val_acc": val_acc,
            }, checkpoint_path)
        else:
            patience_left -= 1

        scheduler.step()
        history.append({
            "backbone": backbone_name,
            "epoch": epoch,
            "train_loss": train_loss,
            "val_logloss": val_logloss,
            "val_acc": val_acc,
            "improved": improved,
            "patience_left": patience_left,
        })
        print(f"[{backbone_name}] epoch={epoch} train_loss={train_loss:.4f} val_logloss={val_logloss:.4f} val_acc={val_acc:.4f} best={best_logloss:.4f}")
        if patience_left < 0:
            break

    history_df = pd.DataFrame(history)
    history_path = EDA_DIR / f"mv_caf_backbone_test_{safe_name}_history_{CUR_VERSION}.csv"
    history_df.to_csv(history_path, index=False)

    checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
    best_state = checkpoint["ema_state_dict"] if CFG["EMA_USE_FOR_EVAL"] else checkpoint["model_state_dict"]
    model.load_state_dict(best_state)
    model.eval()

    tta_df = evaluate_tta_grid(model, val_loader, device, CFG["TTA_CANDIDATES"])
    best_tta = tta_df.iloc[0]
    test_probs, _, test_ids = predict_probs(model, test_loader, device, tta_names=best_tta["tta_names"])

    sub = pd.DataFrame({"id": test_ids, "unstable_prob": test_probs})
    sub["stable_prob"] = 1.0 - sub["unstable_prob"]
    submission_path = SUBMISSION_DIR / f"mv_caf_backbone_test_{safe_name}_{CUR_VERSION}.csv"
    sub.to_csv(submission_path, index=False, encoding="utf-8-sig")

    return {
        "backbone": backbone_name,
        "img_size": img_size,
        "batch_size": batch_size,
        "accum_steps": accum_steps,
        "best_epoch": int(checkpoint["epoch"]),
        "best_val_logloss": float(checkpoint["val_logloss"]),
        "best_val_acc": float(checkpoint["val_acc"]),
        "tta_logloss": float(best_tta["val_logloss"]),
        "tta_acc": float(best_tta["val_acc"]),
        "best_tta": best_tta["tta_names"],
        "checkpoint_path": str(checkpoint_path),
        "submission_path": str(submission_path),
        "history_path": str(history_path),
    }


In [7]:
results = [train_single_backbone(name) for name in selected_backbones]
dino_eval_df = pd.DataFrame(results)
dino_eval_path = EDA_DIR / f"mv_caf_backbone_test_dino_only_{CUR_VERSION}_eval.csv"
dino_eval_df.to_csv(dino_eval_path, index=False)
print("saved:", dino_eval_path)
print(dino_eval_df.to_string(index=False))


[vit_base_patch14_dinov2.lvd142m] using img_size=518


Training:   0%|          | 0/63 [00:02<?, ?it/s]


OutOfMemoryError: CUDA out of memory. Tried to allocate 66.00 MiB. GPU 0 has a total capacity of 23.55 GiB of which 24.31 MiB is free. Process 4335 has 260.00 MiB memory in use. Including non-PyTorch memory, this process has 23.05 GiB memory in use. Of the allocated memory 22.33 GiB is allocated by PyTorch, and 401.90 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
prev_eval_path = PREV_EDA_DIR / f"mv_caf_backbone_test_{PREV_VERSION}_eval.csv"
prev_eval_df = pd.read_csv(prev_eval_path)
combined_eval_df = pd.concat([prev_eval_df, dino_eval_df], ignore_index=True)
combined_eval_df = combined_eval_df.sort_values(["tta_logloss", "tta_acc"], ascending=[True, False]).reset_index(drop=True)
combined_eval_path = EDA_DIR / f"mv_caf_backbone_test_{CUR_VERSION}_eval.csv"
combined_eval_df.to_csv(combined_eval_path, index=False)
print("saved:", combined_eval_path)
print(combined_eval_df.to_string(index=False))

prev_sub_paths = sorted(p for p in PREV_SUBMISSION_DIR.glob("*.csv") if "ensemble" not in p.name)
ensemble_frames = []
for p in prev_sub_paths:
    df = pd.read_csv(p, encoding="utf-8-sig").sort_values("id").reset_index(drop=True)
    ensemble_frames.append(df[["id", "unstable_prob"]].rename(columns={"unstable_prob": p.stem}))

dino_sub_path = Path(results[0]["submission_path"])
dino_sub = pd.read_csv(dino_sub_path, encoding="utf-8-sig").sort_values("id").reset_index(drop=True)
ensemble_frames.append(dino_sub[["id", "unstable_prob"]].rename(columns={"unstable_prob": dino_sub_path.stem}))

ensemble_df = ensemble_frames[0]
for frame in ensemble_frames[1:]:
    ensemble_df = ensemble_df.merge(frame, on="id", how="inner")

prob_cols = [c for c in ensemble_df.columns if c != "id"]
ensemble_sub = pd.DataFrame({"id": ensemble_df["id"]})
ensemble_sub["unstable_prob"] = ensemble_df[prob_cols].mean(axis=1)
ensemble_sub["stable_prob"] = 1.0 - ensemble_sub["unstable_prob"]
ensemble_path = SUBMISSION_DIR / f"mv_caf_backbone_test_ensemble_{CUR_VERSION}.csv"
ensemble_sub.to_csv(ensemble_path, index=False, encoding="utf-8-sig")
print("saved:", ensemble_path)
